# Two-Layer RNN

This tutorial applies the same compilation and inspection workflow to a stacked recurrent network. The additional layer introduces a second hidden group and more parameter-to-hidden relations, making inclusion and exclusion decisions easier to compare.

## Define and Compile the Model

Two `GRUCell` layers are stacked before a linear readout. Compilation, rather than layer width or naming, is the authoritative source for how their hidden states and ETP parameters are grouped.

In [1]:
import jax
import jax.numpy as jnp
import brainstate
import braintrace

In [2]:
class TwoLayerRNN(brainstate.nn.Module):
    def __init__(self, n_in, n_rec, n_out):
        super().__init__()
        self.rnn1 = braintrace.nn.GRUCell(n_in, n_rec)
        self.rnn2 = braintrace.nn.GRUCell(n_rec, n_rec)
        self.out = braintrace.nn.Linear(n_rec, n_out)

    def update(self, x):
        h1 = self.rnn1(x)
        h2 = self.rnn2(h1)
        return self.out(h2)


model2 = TwoLayerRNN(10, 32, 5)

# Compile for a single unbatched sample (no batch_size).
learner2 = braintrace.compile(model2, braintrace.D_RTRL, jnp.zeros(10))
learner2.show_graph()

The hidden groups are:

   Group 0: [('rnn1', 'h')]
   Group 1: [('rnn2', 'h')]


The weight parameters which are associated with the hidden states are:

   Weight 0: ('rnn1', 'Wz', 'weight')  is associated with hidden group 0
   Weight 1: ('rnn1', 'Wh', 'weight')  is associated with hidden group 0
   Weight 2: ('rnn2', 'Wz', 'weight')  is associated with hidden group 1
   Weight 3: ('rnn2', 'Wh', 'weight')  is associated with hidden group 1


The non-etrace weight parameters are:

   Weight 0: ('out', 'weight')  (excluded: relation_excluded_non_temporal)
   Weight 1: ('rnn1', 'Wr', 'weight')  (excluded: relation_excluded_weight_to_weight)
   Weight 2: ('rnn2', 'Wr', 'weight')  (excluded: relation_excluded_weight_to_weight)





## Read the Compiler Output

The compiler reports two hidden groups: Group 0 contains `('rnn1', 'h')`, and Group 1 contains `('rnn2', 'h')`. Equal hidden widths do not merge sequential recurrent layers; the discovered dataflow determines grouping.

For each layer, the update-gate (`Wz`) and candidate (`Wh`) weights have direct ETP relations to that layer's hidden group. The reset-gate (`Wr`) weights are excluded because their path reaches a hidden state only through another trainable ETP primitive (`relation_excluded_weight_to_weight`). The readout is excluded for the distinct non-temporal reason seen in the single-layer model.

## Inspect Both Hidden Groups

`learner2.graph` provides the relation-level representation, while `learner2.report.counts` provides a compact consistency check. Reading them together answers three separate questions: how many recurrent groups were found, which parameter relations were retained, and why other trainable paths were excluded.

In [3]:
# Inspect the two-layer graph programmatically via learner.graph and learner.report
graph2 = learner2.graph

print(f"Number of hidden groups: {len(graph2.hidden_groups)}")
print(f"Number of weight-hidden relations: {len(graph2.hidden_param_op_relations)}")

print("\nHidden groups:")
for g in graph2.hidden_groups:
    print(f"  Group {g.index}: {g.hidden_paths}")

print("\nRelations:")
for i, r in enumerate(graph2.hidden_param_op_relations):
    groups = [g.index for g in r.hidden_groups]
    # ``trainable_paths`` maps each trainable key to its owning ParamState path.
    print(f"  Weight {i}: {r.trainable_paths} -> hidden group(s) {groups}")

# Quick summary via the report
print("\nReport counts:", learner2.report.counts)

Number of hidden groups: 2
Number of weight-hidden relations: 4

Hidden groups:
  Group 0: [('rnn1', 'h')]
  Group 1: [('rnn2', 'h')]

Relations:
  Weight 0: {'weight': ('rnn1', 'Wz', 'weight'), 'bias': ('rnn1', 'Wz', 'weight')} -> hidden group(s) [0]
  Weight 1: {'weight': ('rnn1', 'Wh', 'weight'), 'bias': ('rnn1', 'Wh', 'weight')} -> hidden group(s) [0]
  Weight 2: {'weight': ('rnn2', 'Wz', 'weight'), 'bias': ('rnn2', 'Wz', 'weight')} -> hidden group(s) [1]
  Weight 3: {'weight': ('rnn2', 'Wh', 'weight'), 'bias': ('rnn2', 'Wh', 'weight')} -> hidden group(s) [1]

Report counts: {'hidden_groups': 2, 'etrace_weights': 8, 'excluded_weights': 3, 'warnings': 3, 'errors': 0}


## Compare with the Single-Layer RNN

| Property | Single layer | Two layers |
|---|---:|---:|
| Recurrent hidden groups | 1 | 2 |
| Recurrent layer paths | `rnn` | `rnn1`, `rnn2` |
| Relation interpretation | One temporal recurrence | Per-layer temporal recurrences |
| Non-temporal readout | Excluded | Excluded |

The important change is structural rather than cosmetic. Adding a recurrent layer creates another state transition that the compiler must represent independently. The report explains classification outcomes, and the graph identifies the exact parameter, primitive, and hidden-group connections behind those outcomes.

## Summary

The two-layer model uses the same workflow as the single-layer model: compile, inspect the report, and verify the graph relations. The richer result demonstrates why compiler diagnostics must be interpreted at the relation level rather than inferred from module names or parameter counts. These structural checks still do not replace numerical gradient validation.